# JEPA-SpatioTemporal (Phase 3b) — Pretraining Notebook

**Masking Strategy:** Dual-axis masking: multi-block temporal masking plus zeroing one randomly chosen field group (BidPrice, AskPrice, BidVolume, AskVolume) on 30% of temporally visible timesteps.

**Key Specifications:**
- **Backbone:** SimLOB `FCN1 -> Transformer stack` (no `FCN2` / `reduce_proj`). Exactly 1,064,704 parameters.
- **Predictor:** 4-layer Transformer (`d_model=256`, `nhead=8`, `dim_feedforward=512`).
- **Target Encoder:** Momentum-updated via EMA (0.996 -> 1.0 linear schedule). No backprop.
- **Loss:** Masked-only MSE prediction loss + VICReg variance/covariance collapse prevention (lambda=1.0).
- **Data:** Joint 5-stock pooled dataset (stride 1, leakage-safe session boundaries).
- **Epochs:** 100, Adam optimizer (lr=1e-4), batch size 256, num_workers=2.
- **Checkpoints:** Saved to `jepa_checkpoints/3b/` (save_top_k=1, save_last=True).


In [ ]:
# 1. Environment Setup & Google Drive Mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

!pip install -q lightning pandas numpy torch scikit-learn
print('Environment and packages ready.')


In [ ]:
# 2. Imports, Deterministic Seed & Hardware Inspection
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from common import set_seed
from jepa_common import (
    JEPABackbone,
    Predictor,
    JEPALightningModule,
    prepare_pooled_datasets,
    select_optimal_checkpoint,
    JEPAEncoderForEval,
    ALL_STOCKS,
    LAMBDA_COLLAPSE,
)

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import ModelCheckpoint
    from lightning.pytorch.loggers import CSVLogger
    from lightning.pytorch import Trainer
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint
    from pytorch_lightning.loggers import CSVLogger
    from pytorch_lightning import Trainer

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


In [ ]:
# 3. Mandatory Parameter Count Verification (Section 1 Check)
backbone = JEPABackbone()
backbone_params = sum(p.numel() for p in backbone.parameters())
predictor = Predictor()
predictor_params = sum(p.numel() for p in predictor.parameters())

print(f'JEPABackbone Parameter Count: {backbone_params:,}')
print(f'Predictor Parameter Count:    {predictor_params:,}')

# SimLOB Phase 1 encoder was 5,828,136 params. FCN2 (4,753,152) + reduce_proj (10,280) = 4,763,432.
# 5,828,136 - 4,763,432 = 1,064,704 exact parameters.
assert backbone_params == 1064704, f'Backbone params {backbone_params} != 1064704'
print('✓ Parameter count check PASSED: Exactly matches SimLOB encoder minus FCN2/reduce_proj.')


In [ ]:
# 4. Prepare Joint Pooled 5-Stock Dataset (Section 5)
print('Preparing pooled 5-stock dataset for pretraining...')
train_ds, val_ds = prepare_pooled_datasets(stocks=ALL_STOCKS, data_dir='data', seq_len=100)

batch_size = 256
num_workers = 2
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

print(f'Train Batches: {len(train_loader):,} | Val Batches: {len(val_loader):,}')


In [ ]:
# 5. Configure Training Module & Checkpointing for Variant 3b
ckpt_dir = 'jepa_checkpoints/3b'
log_dir = 'jepa_training_logs/3b'
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

epochs = 100
total_steps = epochs * len(train_loader)

model = JEPALightningModule(
    variant='3b',
    lr=1e-4,
    total_steps=total_steps,
    lambda_collapse=LAMBDA_COLLAPSE,
)

checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename='best',
    save_top_k=1,
    monitor='val_loss',
    mode='min',
    save_last=True,
)
logger = CSVLogger(save_dir=log_dir, name='')

# Check for existing checkpoint to resume
last_ckpt = os.path.join(ckpt_dir, 'last.ckpt')
ckpt_path = last_ckpt if os.path.exists(last_ckpt) else None
if ckpt_path:
    print(f'Found existing checkpoint at {ckpt_path}. Resuming training...')
else:
    print('Starting fresh training from epoch 0...')

trainer = Trainer(
    max_epochs=epochs,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback],
    logger=logger,
    enable_progress_bar=True,
    deterministic=True,
)


In [ ]:
# 6. Execute Training Loop
print('Starting JEPA pretraining...')
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader, ckpt_path=ckpt_path)
print('Training complete.')


In [ ]:
# 7. Non-Collapse Evidence Verification (Section 9 Evidence Standard)
best_ckpt = select_optimal_checkpoint(ckpt_dir, metric_name='val_loss')
print(f'\nBest Checkpoint Selected: {best_ckpt}')

ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
prefix = 'context_encoder.'
backbone_dict = {k[len(prefix):]: v for k, v in ckpt['state_dict'].items() if k.startswith(prefix)}
eval_backbone = JEPABackbone()
eval_backbone.load_state_dict(backbone_dict)
eval_encoder = JEPAEncoderForEval(eval_backbone).to(device)
eval_encoder.eval()

# Check standard deviation across representation dimensions on validation batch
with torch.no_grad():
    sample_x = next(iter(val_loader)).to(device)
    raw_context = eval_backbone(sample_x)  # [B, 100, 256]
    z_flat = raw_context.reshape(-1, 256)
    dim_std = torch.sqrt(z_flat.var(dim=0) + 1e-4)
    print(f'Representation Standard Deviation across 256 dims on validation batch:')
    print(f'  Mean std: {dim_std.mean().item():.4f}')
    print(f'  Min std:  {dim_std.min().item():.4f}')
    print(f'  Max std:  {dim_std.max().item():.4f}')
    assert dim_std.mean().item() > 0.05, 'WARNING: Mean representation std is near zero (collapse)!'
    print('✓ Non-collapse check PASSED: Encoder outputs active, high-variance representations.')
